# KorQuAD를 Chat SFT 데이터로 변환하기

KorQuAD(Korean Question Answering Dataset) v1.0은 한국어 문맥, 질문, 정답 위치와 정답 텍스트를 함께 제공하는 기계독해 데이터셋이다.
한 주제 아래 여러 문단이 있고 한 문단 아래 여러 질문과 정답 후보가 있으므로, 바로 대화형 학습에 쓰기 전에 이 **중첩 구조를 평탄화**해야 한다.
**평탄화**는 여러 단계에 나뉜 값을 같은 단위의 질문-정답 레코드 목록으로 펼치는 변환이다.

Chat SFT(Chat Supervised Fine-tuning) 데이터는 `messages` 안에 역할과 내용을 기록한 모범 대화 예시이다. 앞선 말투 데이터 수업에서는 질문과 모범 답변을 직접 작성했지만, 여기서는 KorQuAD의 `answers[0]["text"]`를 모범 답변으로 매핑한다.

**학습 데이터와 평가 데이터의 분리**는 모델이 본 예시와 보지 않은 예시를 나누는 과정이며, 같은 질문이 양쪽에 들어가 성능이 부풀려지는 누수를 막기 위해 필요하다.

## KorQuAD v1.0 개발 세트 내려받기

공개 KorQuAD v1.0 dev URL의 JSON을 현재 작업 폴더에 `KorQuAD_v1.0_dev.json`으로 저장한다. 이 파일은 다음 셀에서 `json.load()`로 읽을 입력이다.

In [1]:
import json
from pathlib import Path
from urllib.request import urlretrieve

KORQUAD_DEV_URL = "https://raw.githubusercontent.com/korquad/korquad.github.io/refs/heads/master/dataset/KorQuAD_v1.0_dev.json"
dataset_path = Path("KorQuAD_v1.0_dev.json")

urlretrieve(KORQUAD_DEV_URL, dataset_path)
print({"path": str(dataset_path), "exists": dataset_path.exists()})

{'path': 'KorQuAD_v1.0_dev.json', 'exists': True}


## 중첩 JSON을 질문-정답 쌍으로 평탄화하기

최상위 `data`에는 주제가, 각 주제의 `paragraphs`에는 문맥과 `qas`가, 각 QA의 `answers`에는 가능한 정답 후보가 들어 있다. 이 실습은 각 QA에서 `question`과 `answers[0]["text"]`만 꺼내 같은 구조의 딕셔너리 목록으로 펼친다. 첫 정답 하나만 선택하므로 다른 정답 표기와 문맥 정보는 학습 레코드에 포함되지 않는다.

질문을 딕셔너리의 key로 바로 합치면 같은 질문이 덮어써질 수 있다. 여기서는 모든 QA를 리스트에 보존한 뒤 중복 여부를 별도 검사한다. 평탄화된 목록의 순서를 유지해야 다음 단계에서 앞의 20건을 같은 기준으로 선택할 수 있다.

In [2]:
with dataset_path.open("r", encoding="utf-8") as file:
    dev_data = json.load(file)

qa_pairs = []
paragraph_count = 0

#  각 제의 문단과 QA를 원래 순서대로 순회한다.
for topic in dev_data["data"]:
    for paragraph in topic["paragraphs"]:
        paragraph_count += 1
        for qa in paragraph["qas"]:
            # answers[0]은 여러 정답 후보 가운데 이번에 사용할 첫 번째 정답이다.
            qa_pairs.append({
                "question": qa["question"],
                "answer": qa["answers"][0]["text"],
            })

# qa_pairs의 앞 20건이 다음 Chat SFT 변환 셀의 입력이 된다.
assert len(qa_pairs) >= 20, "앞의 20건을 선택할 수 있을 만큼 QA가 필요하다."
print({
    "topics": len(dev_data["data"]),
    "paragraphs": paragraph_count,
    "qa_pairs": len(qa_pairs),
    "first_pair": qa_pairs[0],
})


{'topics': 140, 'paragraphs': 964, 'qa_pairs': 5774, 'first_pair': {'question': '임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?', 'answer': '1989년 2월 15일'}}


## 앞의 20건을 Chat SFT 레코드로 바꾸기

평탄화된 순서에서 앞의 20건을 선택하고, 각 질문-정답 쌍을 `messages` 하나를 가진 레코드로 바꾼다. `user`에는 질문을, 마지막 `assistant`에는 모델이 생성하도록 학습할 첫 번째 정답을 넣는다.

`system` 역할은 유지하지만 content는 빈 문자열로 둔다. 이는 질문만으로 답하는 기존 데이터 계약을 보존하면서 별도의 말투나 행동 지시를 추가하지 않기 위한 선택이다. 빈 system도 문자열 content를 가진 메시지이며, 이후 실제 학습과 추론에서 같은 계약을 사용해야 한다. 새로운 system 지시를 넣으려면 학습·평가·추론 데이터를 모두 같은 형식으로 다시 설계해야 한다.

In [3]:

SYSTEM_CONTENT = "항상 시작은 '강사님, 준비되었습니다.', 전반적으로 친절한 챗봇"
selected_pairs = qa_pairs[:20]

def to_chat_sft_record(qa_pair):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_CONTENT},
            {"role": "user", "content": qa_pair["question"]},
            {"role": "assistant", "content": qa_pair["answer"]},
        ]
    }

chat_records = [to_chat_sft_record(qa_pair) for qa_pair in selected_pairs]
assert len(chat_records) == 20, "Chat SFT 레코드는 20건이어야 한다."
print("레코드 수:", len(chat_records))
print(json.dumps(chat_records[0], ensure_ascii=False, indent=2))

레코드 수: 20
{
  "messages": [
    {
      "role": "system",
      "content": "항상 시작은 '강사님, 준비되었습니다.', 전반적으로 친절한 챗봇"
    },
    {
      "role": "user",
      "content": "임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?"
    },
    {
      "role": "assistant",
      "content": "1989년 2월 15일"
    }
  ]
}


## 20건을 학습용 16건과 평가용 4건으로 분리하기

학습용 레코드는 가중치를 조정하는 예시이고, 평가용 레코드는 학습에 사용하지 않고 변환 결과나 모델 성능을 점검하는 홀드아웃 예시이다. 같은 20건과 같은 seed를 사용하면 매번 같은 인덱스가 선택되도록 표준 라이브러리의 독립적인 난수 생성기를 사용한다.

전체 20건의 원래 순서는 `chat_records`에 그대로 유지한다. 분할할 인덱스만 섞은 뒤 각 집합 안에서는 다시 정렬하므로, 파일을 읽을 때 원래 데이터 순서를 추적할 수 있다. 4건은 분할 구조를 익히기 위한 규모이므로 신뢰할 수 있는 성능 평가 규모로 해석하지 않는다.

In [4]:
import random

TRAIN_SIZE = 16
SPLIT_SEED = 42

split_indices = list(range(len(chat_records)))

random.Random(SPLIT_SEED).shuffle(split_indices)
train_indices = sorted(split_indices[:TRAIN_SIZE])
eval_indices = sorted(split_indices[TRAIN_SIZE:])

train_records = [chat_records[index] for index in train_indices]
eval_records = [chat_records[index] for index in eval_indices]
print("train_indices:", train_indices)
print("eval_indices:", eval_indices)


train_indices: [1, 2, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
eval_indices: [0, 3, 7, 8]


## 목적이 다른 세 JSONL 파일 저장하기

JSON Lines(JSONL)은 독립된 JSON 객체 하나를 한 줄에 저장하는 형식이다. 전체·학습·평가 레코드를 각각 저장하고 다시 읽어 줄 수가 유지되는지 확인한다.

- `korquad_data.jsonl`은 기존 파일명과의 호환을 위해 원래 순서의 전체 20건을 저장한다. 이 파일에는 평가 4건도 포함되므로 평가를 분리할 때 학습 파일로 사용하지 않는다.
- `korquad_train.jsonl`은 결정적으로 선택한 학습용 16건을 저장한다.
- `korquad_eval.jsonl`은 학습에서 제외한 평가용 4건을 저장한다.

In [5]:
output_sets = {
    Path("korquad_data.jsonl"): chat_records,
    Path("korquad_train.jsonl"): train_records,
    Path("korquad_eval.jsonl"): eval_records,
}

def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

saved_counts = {}
for path, records in output_sets.items():
    write_jsonl(path, records)

    with path.open("r", encoding="utf-8") as file:
        reloaded_records = [json.loads(line) for line in file if line.strip()]
    assert len(reloaded_records) == len(records), f"{path}의 저장 레코드 수가 다르다."
    saved_counts[str(path)] = len(reloaded_records)

print(saved_counts)


{'korquad_data.jsonl': 20, 'korquad_train.jsonl': 16, 'korquad_eval.jsonl': 4}


## 관리형 OpenAI SFT는 읽기 전용 절차로 확인하기

[OpenAI Supervised fine-tuning 공식 문서](https://developers.openai.com/api/docs/guides/supervised-fine-tuning)는 관리형 fine-tuning 플랫폼을 축소하고 있으며 신규 사용자는 더 이상 접근할 수 없다고 안내한다. 따라서 이 노트북에는 파일 업로드, Job 생성, 상태 조회, 결과 모델 호출 코드를 두지 않는다.

기존 접근 권한이 있는 환경의 절차는 평가 기준 준비 → 학습 JSONL 업로드 → 업로드 파일 ID로 Job 생성 → 상태 확인 → 분리한 평가 데이터로 결과 모델 평가 순서이다. 실제 모델 학습은 `10_sllm_finetuning`의 [LoRA 실습](../10_sllm_finetuning/04_news2stock_analysis_lora_finetuning.ipynb)과 [QLoRA 실습](../10_sllm_finetuning/05_news2stock_analysis_qlora_finetuning.ipynb)으로 이어진다. 플랫폼 상태와 지원 모델은 바뀔 수 있으므로 공식 문서에서 최신 상태를 확인한다.

## JSONL을 이용한 SFT 입력·평가 흐름 체험

`korquad_train.jsonl`의 `user → assistant` 쌍은 모델이 따라야 할 학습 예시이고, `korquad_eval.jsonl`의 `user`는 처음 보는 평가 질문이며 `assistant`는 모델에게 보여 주지 않는 기준 정답이다. 실제 학습 Job을 실행하지 않는 대신, 학습 파일의 16개 예시를 few-shot 문맥으로 넣고 평가 파일의 첫 질문에 답하게 한다. 마지막에 숨겨 둔 정답과 모델 응답을 비교하면 train과 eval JSONL의 역할을 한 번의 API 호출로 확인할 수 있다.

이 실습은 JSONL을 실제로 사용하는 모습을 보여 주지만 SFT 자체는 아니다. few-shot 예시는 요청할 때마다 함께 전송되며 모델 가중치는 바뀌지 않는다. 실제 SFT에서는 같은 학습 예시로 가중치 또는 어댑터를 갱신하고, 평가 정답은 학습에 넣지 않은 채 결과 모델을 검사한다.

출력에서는 평가 질문, 기준 정답, 모델 응답과 두 가지 단순 비교 결과를 확인한다. `exact_match`는 공백과 끝 문장부호를 정리한 두 문자열이 완전히 같은지 나타내고, `reference_in_prediction`은 기준 정답이 더 긴 모델 응답 안에 포함되는지 나타낸다. 두 값은 수업용 확인 기준이며 실제 QA 성능을 대표하는 정식 평가 지표는 아니다. 현재 JSONL은 문맥을 제외했으므로 이 결과를 KorQuAD 기계독해 성능으로 해석하지 않는다.

In [6]:
import os
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 다시 실행한다.")
load_dotenv(dotenv_path, override=False)

client = OpenAI()
text_model = os.getenv("OPENAI_TEXT_MODEL", "gpt-5.6-luna").strip() or "gpt-5.6-luna"

In [7]:

# 앞에서 만든 JSONL 파일을 한 줄씩 읽어 다시 Chat SFT 레코드 목록으로 복원한다.
def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

train_examples = read_jsonl(Path("korquad_train.jsonl"))
eval_examples = read_jsonl(Path("korquad_eval.jsonl"))
assert len(train_examples) == 16 and len(eval_examples) == 4, "학습 16건과 평가 4건이 필요하다."

# system은 한 번만 두고, 학습 레코드의 user 질문과 assistant 정답을 시연 예시로 이어 붙인다.
few_shot_messages = [train_examples[0]["messages"][0].copy()]
for record in train_examples:
    few_shot_messages.extend(
        {"role": message["role"], "content": message["content"]}
        for message in record["messages"][1:]
    )

# 평가 레코드에서는 user 질문만 요청에 넣고 assistant 정답은 채점용 변수로 분리한다.
eval_record = eval_examples[0]
eval_question = eval_record["messages"][1]["content"]
reference_answer = eval_record["messages"][2]["content"]
few_shot_messages.append({"role": "user", "content": eval_question})

# 출력 생성: 16개 학습 예시를 포함한 문맥으로 평가 질문에 한 번만 답하게 한다.
response = client.chat.completions.create(
    model=text_model,
    messages=few_shot_messages,
)
prediction = response.choices[0].message.content.strip()

# 단순 비교에서는 공백과 문자열 끝의 문장부호 차이를 제거
def normalize_short_answer(text):
    return "".join(text.lower().split()).strip(".,!?\"'")

normalized_reference = normalize_short_answer(reference_answer)
normalized_prediction = normalize_short_answer(prediction)
exact_match = normalized_prediction == normalized_reference
reference_in_prediction = normalized_reference in normalized_prediction

print("학습 예시 수:", len(train_examples))
print("평가 질문:", eval_question)
print("기준 정답:", reference_answer)
print("모델 응답:", prediction)
print({"exact_match": exact_match, "reference_in_prediction": reference_in_prediction})


학습 예시 수: 16
평가 질문: 임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배 된 날은?
기준 정답: 1989년 2월 15일
모델 응답: 강사님, 준비되었습니다. 임종석이 여의도 농민 폭력 시위를 주도한 혐의로 지명수배된 날은 **1989년 2월 15일**입니다.
{'exact_match': False, 'reference_in_prediction': True}
